In [1]:
import pandas as pd
import torch
from torch import nn


TREND_THRESHOLD = 1.5
EPOCHS = 2000

torch.manual_seed(42)

In [2]:
close_prices = pd.read_csv("rosneft_close.csv", sep=";")["CLOSE"].tolist()

print("Количество цен:", len(close_prices))
print("Первые десять цен:", close_prices[:10])

Количество цен: 675
Первые десять цен: [527.4, 508.0, 516.9, 510.45, 517.25, 524.15, 531.85, 530.55, 532.25, 532.45]


In [3]:
def calculate_slope(prices):
    periods = [1, 2, 3]
    mean_period = sum(periods) / 3
    mean_price = sum(prices) / 3

    numerator = sum(
        (period - mean_period) * (price - mean_price)
        for period, price in zip(periods, prices)
    )
    denominator = sum((period - mean_period) ** 2 for period in periods)
    return numerator / denominator


def classify_trend(future_prices):
    slope = calculate_slope(future_prices)
    if slope > TREND_THRESHOLD:
        return 1
    elif slope < -TREND_THRESHOLD:
        return -1
    return 0

In [4]:
def make_dataset(prices):
    inputs = []
    classes = []

    for start in range(len(prices) - 12):
        history = prices[start:start + 10]
        future = prices[start + 10:start + 13]


        inputs.append([price - history[-1] for price in history])

        classes.append(classify_trend(future) + 1)

    return (
        torch.tensor(inputs, dtype=torch.float32),
        torch.tensor(classes, dtype=torch.long),
    )

In [5]:
border = int(len(close_prices) * 0.8)
train_inputs, train_classes = make_dataset(close_prices[:border])
test_inputs, test_classes = make_dataset(close_prices[border:])

print("Обучающие входы:", tuple(train_inputs.shape))
print("Обучающие ответы:", tuple(train_classes.shape))
print("Тестовые входы:", tuple(test_inputs.shape))
print("Тестовые ответы:", tuple(test_classes.shape))

Обучающие входы: (528, 10)
Обучающие ответы: (528,)
Тестовые входы: (123, 10)
Тестовые ответы: (123,)


In [6]:
networks = {
    "Сеть 1: 10 → 24 → 3, Tanh": nn.Sequential(
        nn.Linear(10, 24),
        nn.Tanh(),
        nn.Linear(24, 3),
    ),
    "Сеть 2: 10 → 32 → 3, ReLU": nn.Sequential(
        nn.Linear(10, 32),
        nn.ReLU(),
        nn.Linear(32, 3),
    ),
    "Сеть 3: 10 → 24 → 12 → 3, Tanh": nn.Sequential(
        nn.Linear(10, 24),
        nn.Tanh(),
        nn.Linear(24, 12),
        nn.Tanh(),
        nn.Linear(12, 3),
    ),
}

In [7]:
train_losses = {}

for name, network in networks.items():
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(network.parameters(), lr=0.01)

    network.train()
    for epoch in range(EPOCHS):
        optimizer.zero_grad()
        outputs = network(train_inputs)
        loss = loss_function(outputs, train_classes)
        loss.backward()
        optimizer.step()

    train_losses[name] = loss.item()

In [8]:
def get_accuracy(network, inputs, classes):
    network.eval()
    with torch.no_grad():
        predictions = network(inputs).argmax(dim=1)
    return (predictions == classes).float().mean().item()

results = []
for name, network in networks.items():
    results.append({
        "Сеть": name,
        "Ошибка обучения": train_losses[name],
        "Точность на обучении, %": get_accuracy(network, train_inputs, train_classes) * 100,
        "Точность на тесте, %": get_accuracy(network, test_inputs, test_classes) * 100,
    })

pd.DataFrame(results).round(2)

,Сеть,Ошибка обучения,"Точность на обучении, %","Точность на тесте, %"
0,"Сеть 1: 10 → 24 → 3, Tanh",0.47,79.92,29.27
1,"Сеть 2: 10 → 32 → 3, ReLU",0.45,82.95,31.71
2,"Сеть 3: 10 → 24 → 12 → 3, Tanh",0.16,93.37,25.20
